<h1 style="color: #336699;"><b>ELIXIR Spatial course</b></h1>

<h2 style="color: #336699;">Domain identification and niche reconstruction</h2>
<h3 style="color: #336699;">Day 3 - Practical 8</h3>

<h4>8th October 2026</h4>
<h4>Benedetta Manzato (LUMC) & Emma Vonk (LUMC)</h4>

<div style="border: 2px solid #336699; padding: 15px; border-radius: 8px;">
  <span style="color: #336699; font-size: 24px; font-weight: bold;">Agenda:</span>
  <ol style="margin-top: 10px; padding-left: 25px; font-size: 17px; color: #333333;">
    <li>Graph construction and analysis of spatial transcriptomics data using Squidpy</li>
    <li>Spatial Domain detection with CellCharter</li>
    <li>Spatial Clustering with BANKSY</li>
  </ol>
</div>


In [ ]:
import os
import warnings

import anndata as ad
import pandas as pd
import scanpy as sc
import squidpy as sq

warnings.filterwarnings("ignore")

## Dataset: Xenium In Situ Analysis of Alzheimer's Disease Mouse Model Brain Coronal Sections from One Hemisphere Over a Time Course

<font size="3">The dataset consists of 6 coronal mouse brain slices from 2 different conditions (wildtype - ctrl vs TgCRND8 - AD) across 3 timepoints. In this practical, we have information about cell types available in  `adata.obs['cell_types']`. Please note that these annotation are not perfect. For example, there are quite some cells that could not be assigned to a cell type ("Unknown"). These annotations have been made with on leiden clustering and marker genes reported in [this](https://pages.10xgenomics.com/rs/446-PBO-704/images/10x_LIT000210_App-Note_Xenium-In-Situ_Letter_Digital.pdf) document. (<code>adata_loading_and_annotation.ipynb</code>)

<font size="3">In this practical we aim to understand the differences of the mouse brain between the two conditions and across the timepoints using niches and spatial domains.

Download link: https://www.10xgenomics.com/datasets/xenium-in-situ-analysis-of-alzheimers-disease-mouse-model-brain-coronal-sections-from-one-hemisphere-over-a-time-course-1-standard. 


In [ ]:
# load adata
base_dir = '../data/Practical6/'
adata = sc.read_h5ad(f'{base_dir}xenium_mouse_ad_annotated_rotated.h5ad')
adata

In [ ]:
# Creating a DataFrame from 'split', 'fov', and 'condition'
df = adata.obs[['condition', 'time', 'batch_key']]
value_counts = pd.DataFrame(df.values, columns=df.columns).value_counts()
value_counts

## 1. Cell neighborhood detection via graph construction

<font size="3">Spatial transcriptomics data can be represented as graphs with cells as nodes and edges as relations. Depending on the technology (imaging-based or sequencing-based) different assumptions can be made for the graph structure. 
Here we will explore graph construction and cell neighborhood analysis using the squidpy `sq.gr.spatial_neighbors` module on imaging-based data. 

Information for graph reconstruciton for sequencing-based data can be found [here](https://squidpy.readthedocs.io/en/stable/notebooks/examples/graph/compute_spatial_neighbors.html).

<font size="3">For image-based technologies we construct a graph with `coord_type=generic`, meaning that the nodes / cells will preserve their spatial location and will not be re-arranged in a spatial grid. For generic graph approaches we can choose to set a fixed radius `radius` or number of neighbors to connect to (`n_neighs`).

<font size="3">Below we try both graph construction methods and give them a different `key` to add to the `adata` object.

In [ ]:
if 'cell_types_colors' in adata.uns:
    del adata.uns['cell_types_colors']
    
sq.gr.spatial_neighbors(adata, n_neighs=5, coord_type="generic", key_added = 'neighs_based_spatial')
sq.pl.spatial_scatter(
    adata,
    shape=None,
    library_key = 'batch_key',
    color=["celltype"],
    connectivity_key="neighs_based_spatial_connectivities",
    title=adata.obs['batch_key'].cat.categories,
    ncols=3,
    size = 10
)

In [ ]:
sq.gr.spatial_neighbors(adata, radius=0.2, coord_type="generic", key_added = 'radius_based_spatial')
sq.pl.spatial_scatter(
    adata,
    shape=None,
    library_key = 'batch_key', 
    color="celltype",
    connectivity_key="radius_based_spatial_connectivities",
    title=adata.obs['batch_key'].cat.categories,
    ncols=3,
    size=10,
)

<font size="3">First differences across the graphs can be observed by visual inspection, i.e. the cells that are further away from the clear brain structures are connected in the closest neighbor approach but unconnected in the radius-based approach. Some other differences are harder to observe like dependencies in the more dense connected regions. For this Squidpy provides a number of statistics to better the differences between the connectivities of cells.

<font size="3">Using the cell type information we can formulate some hypothesis from the data.

<font size="3">To perform downstream analysis you require a graph structure or radius that connects the cells of interest. For example, if you are interested in the communication between Microglia and Oligodendrocytes you need to set a radius such that the cell typer connections exists.

<font size="3"><strong style="color: #c1121f;">**Task 1:** Compute the [interaction matrix](https://squidpy.readthedocs.io/en/stable/notebooks/examples/graph/compute_interaction_matrix.html) to understand differences between neighborhood and radius based graphs. Find an appropriate number of neighbors or radius that include sufficient connections between your cell types of interest.</span>

<font size="3"><strong style="color: #c1121f;">What are the average number of connections per cell type? Which cell types tend to cluster together? 

In [ ]:
sq.gr.interaction_matrix(adata, cluster_key="celltype", connectivity_key="neighs_based_spatial")
sq.pl.interaction_matrix(adata, cluster_key="celltype", connectivity_key="neighs_based_spatial")

In [ ]:
sq.gr.interaction_matrix(adata, cluster_key="celltype", connectivity_key="radius_based_spatial")
sq.pl.interaction_matrix(adata, cluster_key="celltype", connectivity_key="radius_based_spatial")

<font size="3"><strong style="color: #c1121f;">**Task 2:** Try out some analysis from Squidpy to identify distinctions between the graphs e.i. using the [centrality score](https://squidpy.readthedocs.io/en/stable/notebooks/examples/graph/compute_centrality_scores.html) and [neighborhood enrichment](https://squidpy.readthedocs.io/en/stable/notebooks/examples/graph/compute_nhood_enrichment.html). </span>

In [ ]:
sq.gr.centrality_scores(adata, cluster_key = "celltype", connectivity_key = "neighs_based_spatial")

In [ ]:
sq.pl.centrality_scores(adata, cluster_key = "celltype")

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="celltype", connectivity_key = "radius_based_spatial")

In [ ]:
sq.pl.nhood_enrichment(
    adata, cluster_key="celltype", method="average", figsize=(5, 5)
) 

## 2. Spatial Domain detection with CellCharter

https://www.nature.com/articles/s41588-023-01588-4

In [ ]:
import logging

import cellcharter as cc
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import scvi
import squidpy as sq

logger = logging.getLogger('pytorch_lightning.utilities.rank_zero')
logger.setLevel(logging.ERROR)

In [ ]:
scvi.settings.seed = 12345
scvi.settings.num_threads = 2

<font size="3">There may be cells with very low counts. We will filter them out.

In [ ]:
sc.pp.filter_cells(adata, min_counts=3)

### 2.1 Dimensionality reduction

<font size="3">First, we need to run the dimensionality reduction. We will use [scVI](https://docs.scvi-tools.org/en/latest/api/reference/scvi.model.SCVI.html) for this.

<font size="3">Make sure that `adata.X` contains count data, as it is required by scVI. <br>
In some tutorials you will see that the count data is stored in an AnnData layer called `counts`. Here we will use the `X` layer.

In [ ]:
scvi.model.SCVI.setup_anndata(adata)

In [ ]:
LOAD_MODEL = True

<font size="3">We set the parameters of the neural network. Here we use 1 layer and an embedding size of 10.<br>
<font size="3">These are very common parameters, but you can play with them to see how they affect the results.

In [ ]:
model = scvi.model.SCVI(
        adata,
        n_layers=1,
        n_latent=10,
        use_layer_norm="both",
        use_batch_norm="none",
    )
model.train(early_stopping=True, enable_progress_bar=True, max_epochs=10, accelerator="cpu")

<font size="3">To make sure that the training has converged, we can plot the training history.<br>
<font size="3">Here we is important to focus on the (validation) reconstruction loss. This shows how well the model is able to reconstruct the original data.

In [ ]:
plt.figure(figsize=(5, 5))
plt.plot(
    model.history["reconstruction_loss_train"],
    label="train",
    color="darkgreen",
    linewidth=1.25
)
plt.plot(
    model.history["reconstruction_loss_validation"],
    label="validation",
    color="firebrick",
    linewidth=1.25
    )
plt.legend()
plt.title("reconstruction_loss")  
plt.tight_layout()

<font size="3">We extract the latent representation of the data.

In [ ]:
adata.obsm['X_scVI'] = model.get_latent_representation(adata).astype(np.float32)

### 2.2 Neighborhood aggregation

<font size="3">Until now, all the analyses have been done ignoring the spatial information. <br>
<font size="3">Now we will use the spatial information to perform the clustering.

<font size="3">First, we need to create a network where cells are connected if they are close to each other using the Delaunay triangulation.

In [ ]:
sq.gr.spatial_neighbors(adata, library_key='sample', coord_type='generic')

In [ ]:
sq.pl.spatial_scatter(
    adata, 
    shape=None, 
    library_key='sample',
    library_id=adata.obs['sample'].cat.categories[0],
    color="sample", 
    size=1, 
    figsize=(10,10),
    connectivity_key="spatial_connectivities",
    ncols=1
)

<font size="3">As you can see, the Delaunay triangulation generates very long edges for a few cells. <br>
<font size="3">The most appropriate approach would be to estimate the most biologically relevant distance between cells and use it to remove the edges longer than that distance.

<font size="3">A quick alternative solution is that, since those long edges are sort of outlines, the measure the 99th percentile of the edge lengths and remove the edges longer than that distance. <br>
<font size="3">This process will lead to some isolated cells, but that's not a problem.


In [ ]:
sq.gr.spatial_neighbors(adata, library_key='sample', coord_type='generic', percentile=99)

In [ ]:
sq.pl.spatial_scatter(
    adata, 
    shape=None, 
    library_key='sample',
    library_id=adata.obs['sample'].cat.categories[0],
    color="sample", 
    size=1, 
    figsize=(10,10),
    connectivity_key="spatial_connectivities",
    ncols=1
)

<font size="3">We construct the neighborhood aggregated representation by combining every cell's features with the ones of first 3 layers of neighbors.<br>
<font size="3">We will use the scVI latent representation as features and the new features will be stored in `adata.obsm['X_cellcharter']`.

In [ ]:
cc.gr.aggregate_neighbors(adata, n_layers=3, use_rep='X_scVI', out_key='X_cellcharter_temp', sample_key='sample')

<font size="3"><strong style="color: #c1121f;">**Task 3:** Given that `X_scVI` contains 10 features, how many features does `X_cellcharter` contain?<br>
Guess the answer and then write the code to check it.


In [ ]:
adata.obsm['X_scVI'].shape

In [ ]:
#### check 'X_cellcharter_temp' 

### 2.3 Clustering

<font size="3">Finally, let's cluster the cells and find their spatial domains.<br>
<font size="3">We will use 18 clusters, we will see later why this is a good choice.

In [ ]:
gmm = cc.tl.Cluster(
    n_clusters=18,
    random_state=12345,
    trainer_params={'accelerator': 'cpu'},
)
gmm.fit(adata, use_rep='X_cellcharter_temp')
adata.obs['spatial_domain_temp'] = gmm.predict(adata, use_rep='X_cellcharter_temp')

In [ ]:
adata.write_h5ad(f"{base_dir}adata_with_spatial_domains.h5ad", compression="gzip")

<font size="3">We can now plot domains and cell types back to back

In [ ]:
if 'spatial_domain_temp_colors' in adata.uns:
    del adata.uns['spatial_domain_temp_colors']

sq.pl.spatial_scatter(
    adata, 
    shape=None, 
    library_key='sample', 
    color=["spatial_domain_temp", "celltype"], 
    size=1,
    figsize=(10,10),
    title=np.repeat(adata.obs['sample'].cat.categories, 2),
    ncols=2
)

### 2.4 Downstream analyses

<font size="3">Now that we have the spatial domains, we can look at the cell type enrichment in each domain.

<font size="3">This function measures the likelihood of a cell type being found in a domain compared to random chance.

In [ ]:
cc.gr.enrichment(
    adata,
    group_key='spatial_domain_temp',
    label_key='celltype',
)
cc.pl.enrichment(
    adata,
    group_key='spatial_domain_temp',
    label_key='celltype',
    dot_scale=8
)

<font size="3"><strong style="color: #c1121f;">**Task 4:**
The [Allen Brain Atlas](https://atlas.brain-map.org/atlas?atlas=1&plate=100960076#atlas=1&plate=100960520&resolution=6.98&x=5512.001546223959&y=3967.997233072917&zoom=-2) provides images and annotation of mouse brain samples at multiple depths.<br><br>
Choose one of the six samples, go to the atlas and try to find the image that better matches the regions shown by CellCharter.</span>

<span style="color: #c1121f;;">1. Which of the 132 images is the closest one?</span><br>
<span style="color: #c1121f;;">2. After identifying the closest image, try to match the domains obtained with the annotated regions in the atlas.</span><br>
<span style="color: #c1121f;;">3. Do you find anatomical differences between the two conditions? are there differences in spatial domain compositions? Are there unique domains in one of the two conditions (Alzheimer vs wildtype)?</span><br>
<span style="color: #c1121f;;">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- Focus, in particular on comparing samples from mice of the same age.</span><br>
<span style="color: #c1121f;;">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- If you find difference between the two samples, what do you think is the cause?</span><br>


### Shape characterization

<font size="3">Now, we are going to look at the shapes of the domains we obtained.

<font size="3">We first find the local components of the domains.
<font size="3">This is done by first finding the connected components of the spatial graph.

In [ ]:
adata.obs['spatial_domain_17'] = (adata.obs["spatial_domain_temp"] == '17')

In [ ]:
cc.gr.connected_components(adata, cluster_key='spatial_domain_17', min_cells=100)

In [ ]:
# Hackfix: squidpy's spatial_scatter has some issues with categorical data with NaNs.
adata.obs['component_tmp'] = adata.obs['component'].astype('str')

In [ ]:
if 'component_tmp_colors' in adata.uns:
    del adata.uns['component_tmp_colors']

sq.pl.spatial_scatter(
    adata[(adata.obs['sample'].isin(['TgCRND8_17_9', 'TgCRND8_2_5']))], 
    shape=None, 
    library_key='sample', 
    color=["component_tmp", "spatial_domain_17"], 
    size=1,
    figsize=(10,10),
    title=np.repeat(['TgCRND8_17_9', 'TgCRND8_2_5'], 2),
    ncols=2
)

<font size="3">Then draw a boundary around every component

In [ ]:
cc.tl.boundaries(adata, alpha_start=10)

In [ ]:
cc.pl.boundaries(adata, sample='wildtype_5_7', show_cells=True)

<font size="3">And finally, by computing some shape metrics that we will use to compare the domains.

In [ ]:
cc.tl.curl(adata)
cc.tl.linearity(adata)

<font size="3">Let's look at the shape metrics of domain 8.

In [ ]:
adata.uns['shape_component']

<font size="3"> Plot curl and linearity for spatial domain 15 in the 2 conditions:

In [ ]:
import matplotlib.pyplot as plt

conds = list(adata.obs['condition'].unique())
clusters = [int(k) for k in sorted(adata.uns['shape_component']['curl'].keys(), key=int)][-3:]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # blue, orange, green
x = range(len(conds))

plt.figure(figsize=(8,3), dpi=300)

# --- Curl ---
plt.subplot(1,2,1)
for c, col in zip(clusters, colors):
    val = float(adata.uns['shape_component']['curl'][str(c)])
    plt.scatter(x, [val]*len(conds), s=80, color=col, label=f'Cluster {c}')
plt.xticks(x, conds, rotation=45)
plt.title(f"Curl — clusters {clusters}")
plt.ylabel("curl")
plt.legend(frameon=False, fontsize=8)

# --- Linearity ---
plt.subplot(1,2,2)
for c, col in zip(clusters, colors):
    val = float(adata.uns['shape_component']['linearity'][str(c)])
    plt.scatter(x, [val]*len(conds), s=80, color=col, label=f'Cluster {c}')
plt.xticks(x, conds, rotation=45)
plt.title("Linearity — clusters 9, 10, 15")
plt.ylabel("linearity")
plt.legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.show()


<font size="3"><strong style="color: #c1121f;">**Task 6:** Look at the spatial domain 9.</span><br>
<span style="color: #c1121f;;">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- Is it going to have lower, same or higher linearity than domain 9?</span><br>
<span style="color: #c1121f;;">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- What about curl?</span>

<font size="3"><span style="color: #c1121f;;">After you have answered, plot the shape metrics for domain 9 and check if it matches your expectations.</span>

<font size="3"><strong style="color: #c1121f;">**Task 7:** Look at spatial domain 10.</span><br>
<span style="color: #c1121f;;">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- What do you expect to see in terms of curl and linearity?</span><br>
<span style="color: #c1121f;;">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- What about elongation?</span>

<font size="3"><span style="color: #c1121f;;">After answering, compute the elongation and compare the shape metrics for domains 10.</span>

<font size="3">You can check [CellCharter's documentation](https://cellcharter.readthedocs.io/en/latest/tools.html) for the elongation metric.


In [ ]:
conds = list(adata.obs['condition'].unique())
curl_val = float(adata.uns['shape_component']['curl'][str(clusters[-1])])
lin_val  = float(adata.uns['shape_component']['linearity'][str(clusters[-1])])
x = range(len(conds))

plt.figure(figsize=(8,3), dpi=300)

plt.subplot(1,2,1)
plt.scatter(x, [curl_val]*len(conds), s=80)
plt.xticks(x, conds, rotation=45)
plt.title("Curl — cluster 15")
plt.ylabel("curl")

plt.subplot(1,2,2)
plt.scatter(x, [lin_val]*len(conds), s=80)
plt.xticks(x, conds, rotation=45)
plt.title("Linearity — cluster 15")
plt.ylabel("linearity")

plt.tight_layout()
plt.show()

In [ ]:
conds = list(adata.obs['condition'].unique())
clusters = [int(k) for k in sorted(adata.uns['shape_component']['curl'].keys(), key=int)][-3:]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # blue, orange, green
x = range(len(conds))

plt.figure(figsize=(8,3), dpi=300)

# --- Curl ---
plt.subplot(1,2,1)
for c, col in zip(clusters, colors):
    val = float(adata.uns['shape_component']['curl'][str(c)])
    plt.scatter(x, [val]*len(conds), s=80, color=col, label=f'Cluster {c}')
plt.xticks(x, conds, rotation=45)
plt.title(f"Curl — clusters {clusters}")
plt.ylabel("curl")
plt.legend(frameon=False, fontsize=8)

# --- Linearity ---
plt.subplot(1,2,2)
for c, col in zip(clusters, colors):
    val = float(adata.uns['shape_component']['linearity'][str(c)])
    plt.scatter(x, [val]*len(conds), s=80, color=col, label=f'Cluster {c}')
plt.xticks(x, conds, rotation=45)
plt.title("Linearity — clusters 9, 10, 15")
plt.ylabel("linearity")
plt.legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import gc

gc.collect()

## 3. BANKSY

<font size="3">In this notebook we are using the Python implementation of [BANSKY](https://github.com/prabhakarlab/Banksy_py). 

<font size="3">Let's reload the data to restore the original version.

In [ ]:
# load adata again
adata = ad.read_h5ad(f'{base_dir}xenium_mouse_ad_annotated_rotated.h5ad')
adata

<font size="3">We run BANKSY on a example section, select a section that you find interesting.

In [ ]:
adata_section = adata[(adata.obs['time'] == '5_7') & (adata.obs['condition'] == 'wildtype')]
adata_section

In [ ]:
del adata
gc.collect()

In [ ]:
## add x and y coordinate to .obs (needed for plot later)
sp = adata_section.obsm.get('spatial')
sp_df = pd.DataFrame(sp, index=adata_section.obs_names, columns=['x','y'])
adata_section.obs[['x','y']] = sp_df.loc[adata_section.obs_names, ['x','y']]

In [ ]:
from banksy_utils.filter_utils import normalize_total

# Normalizes the AnnData object
adata_section = normalize_total(adata_section)

<font size="3">BANSKY unifies spatially informed cell type and domain segmentation. In this notebook we will focus on the domain segmentation part. 

<font size="3">The idea is to create a representation of a cell using 1) its own transcriptomic profile (purple) and 2) local microenvironment (red + light pink). The local microenvironment is represented through a pair of spatial kernels that represent the mean gene expression of the local microenvironment (red) and its gradient calculated with the azimuthal Gabor filter (AGF) (light pink). 

<font size="3">The relative contribution of the microenvironment is captured by $\lambda$. Smaller settings of $\lambda$ decrease the influence of the cells within the microenvironment ($\lambda = 0$ reduces to nonspatial informed clustering). $G(r)$ is a radially symmetric Gaussian kernel that decays from magnitude 1 at distance = 0.

<font size="3">The main BANSKY algorithm requires: 

<font size="3">1. Creating a kNN graph by setting a spatial number of neighbords `num_neighbors` (`k_geom`) parameter. 
<font size="3">2. Assigning weights to the edges of the conected spatial graph. By default, we use the `gaussian decay` option, where weights decay as a function of distance to the index cell with $\sigma$ `= sigma`.
<font size="3">3. Defining whether to use the Azumithal Gabor Filter kernel (`max_m = 1`) or just the mean expression (`max_m = 0`).

<font size="3">First, we set the required parameters.

In [ ]:
coord_keys = ('x', 'y', 'spatial')

# set parameters 
plot_graph_weights = True
k_geom = 15 # number of neighbors
max_m = 1 # azumithal transform up to kth order
nbr_weight_decay = "scaled_gaussian" # can also be "reciprocal", "uniform" or "ranked"

### Construct the $k_{geom}$-NN graph

In [ ]:
from banksy.main import median_dist_to_nearest_neighbour

# Find median distance to closest neighbours, the median distance will be `sigma`
nbrs = median_dist_to_nearest_neighbour(adata_section, key = coord_keys[2])

### Generate spatial weights from distance

<font size="3">Here, we generate the spatial weights using the gaussian decay function from the median distance to the k-th nearest neighbours as specified earlier.

In [ ]:
from banksy.initialize_banksy import initialize_banksy

plt.style.use('default')

banksy_dict = initialize_banksy(
    adata_section,
    coord_keys,
    k_geom,
    nbr_weight_decay=nbr_weight_decay,
    max_m=max_m,
    plt_edge_hist=True,
    plt_nbr_weights=True,
    plt_agf_angles=False, # takes long time to plot
    plt_theta=True,
)

### Generate BANKSY matrix

<font size="3">The BANKSY matrix considers a cells transcriptomic profile and local microenvironment (Figure 1). 

<font size="3">As mentioned before, $\lambda$ is a mixing parameter that controls the importance of cells’ own expression and neighborhood expression effects, it takes values from 0, being spatial information not used in the clustering, to 1, giving the maximum importance to the neighborhood expression.

<font size="3">To generate the BANKSY matrix, we proceed with the following:

<font size="3">1. Matrix multiply sparse CSR weights matrix with cell-gene matrix to get **neighbourhood matrix** and the **AGF matrix** if `max_m > 1`

<font size="3">2. Z-score both matrices along **genes**

<font size="3">3. Multiply each matrix by a weighting factor $\lambda$ (We refer to this parameter as lambda in our manuscript and code)

<font size="3">4. Concatenate the matrices along the genes dimension in the form -> `horizontal_concat(cell_mat, nbr_mat, agf_mat)`

<font size="3">Here, we save all the results in the dictionary (banksy_dict), which contains the results from the subsequent operations for BANKSY.

In [ ]:
from banksy.embed_banksy import generate_banksy_matrix

# The following are the main hyperparameters for BANKSY
lambda_list = [0.6] # list of lambda parameters, can be multiple! we will run one for time efficiency

banksy_dict, banksy_matrix = generate_banksy_matrix(adata_section, banksy_dict, lambda_list, max_m)

In [ ]:
from banksy.main import concatenate_all

banksy_dict["nonspatial"] = {
    # Here we simply append the nonspatial matrix (adata.X) to obtain the nonspatial clustering results
    0.0: {"adata": concatenate_all([adata_section.X], 0, adata=adata_section), }
}

print(banksy_dict['nonspatial'][0.0]['adata'])

### Reduce dimensions of each data matrix

<font size="3">We utilize two common methods for dimensionality reduction:

<font size="3">1. PCA (using `scikit-learn`), we reduce the size of thematrix from $3 * N_{genes}$ to `pca_dims`. As a default settings, we reduce to 20 dimensions.

<font size="3">1. UMAP (`UMAP` package), which we use to visualize expressions of clusters in the umap space (2-D space).

In [ ]:
## Define hyperparameters

resolutions = [0.1] # clustering resolution for UMAP
pca_dims = [20] # Dimensionality in which PCA reduces to 20

In [ ]:
from banksy_utils.umap_pca import pca_umap

pca_umap(banksy_dict,
         pca_dims = pca_dims,
         add_umap = True,
         plt_remaining_var = False,
         )

### Cluster cells using a partition algorithm

<font size="3">We then cluster cells using the **leiden** algorithm partition methods. Other clustering algorithms include *louvain* (another resolution based clustering algorithm), or *mclust* (a clustering based on gaussian mixture model).

In [ ]:
from banksy.cluster_methods import run_Leiden_partition

seed = 0
results_df, max_num_labels = run_Leiden_partition(
    banksy_dict,
    resolutions,
    num_nn = 50,
    num_iterations = -1,
    partition_seed = seed,
    match_labels = True,
)

### Plot results

In [ ]:
from banksy.plot_banksy import plot_results

c_map =  'tab20' # specify color map
weights_graph =  banksy_dict['scaled_gaussian']['weights'][0]

In [ ]:
import matplotlib as mpl
from matplotlib import cm

banksy_path = 'output/'

# Patch get_cmap back onto cm for libraries using the old API
if not hasattr(cm, 'get_cmap'):
    cm.get_cmap = mpl.colormaps.get_cmap # NOTE: code shown here has been added to prevent errors with the newer matplotlib versions

plot_results(
    results_df,
    weights_graph,
    c_map,
    match_labels = True,
    coord_keys = coord_keys,
    max_num_labels  =  max_num_labels, 
    save_path = os.path.join(banksy_path, 'tmp_png'),
    save_fig = True, # save the spatial map of all clusters
    save_seperate_fig = True, # save the figure of all clusters plotted seperately
)

<font size="3"><strong style="color: #c1121f;"> If UMAP-PCA computation takes too long read the results df:

In [ ]:
adata_banksy = ad.read_h5ad(f'{base_dir}adata_banksy_results.h5ad')
adata_banksy

<font size="3"> Plot the resulting spatial domains:

In [ ]:
# Extract x/y coordinates and domain labels from .obs
x = adata_banksy.obs['x']
y = adata_banksy.obs['y']
domains = adata_banksy.obs['labels_scaled_gaussian_pc20_nc0.60_r0.20']

# Plot
plt.figure(figsize=(6,6), dpi=200)
plt.scatter(
    x, y,
    c=domains.astype('category').cat.codes,
    cmap='tab20',
    s=0.5,
    alpha=0.8
)
plt.axis('equal')
plt.xticks([])
plt.yticks([])
plt.title("Spatial Domains (BANKSY)", fontsize=10)
plt.show()

### Investigate Cell Type composition in each Banksy-defined Spatial Domain

In [ ]:
def plot_sd_vs_cell_type_composition(res_df,idx):
    """
    Plots the cell type composition as a percentage across different SD (standard deviation) values.
    The data is visualized as a stacked bar plot.

    Parameters:
    - results_df_lambda05: DataFrame containing the data with columns 'labels_scaled_gaussian_pc20_nc0.50_r0.10', 'class', and others.
    - idx: string of column of interest in the DataFrame
    """
    # Step 1: Add a 'Count' column to facilitate pivoting (each row contributes a count of 1)
    res_df.obs['Count'] = 1

    # Step 2: Create a pivot table with SD as the index, cell types as columns, and the sum of counts as values
    pivot_df = res_df.obs.pivot_table(
        index=idx,  # Group by SD
        columns='celltype', 
        values='Count', 
        aggfunc='sum',  
        fill_value=0  
    )

    # Ensure SD values are numeric
    pivot_df.index = pivot_df.index.astype(float)

    # Step 3: Convert counts to percentages for each SD
    pivot_df = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

    # Step 4: Set up the plot
    _, ax = plt.subplots(figsize=(10, 6)) 

    # Plot stacked bars
    bottom = None  
    for cell_type in pivot_df.columns: 
        ax.bar(
            pivot_df.index, 
            pivot_df[cell_type],  
            label=cell_type,  
            bottom=bottom  
        )
        bottom = pivot_df[cell_type] if bottom is None else bottom + pivot_df[cell_type]

    # Step 5: Add labels and title
    ax.set_xlabel('SD')  
    ax.set_ylabel('Cell Type Composition (%)')  
    ax.set_title('SD vs. Cell Type Composition')  
    ax.set_ylim(0, 100)  

    # Add legend
    plt.legend(
        title="Cell Type",  
        bbox_to_anchor=(1.05, 1),  
        loc='upper left' 
    )

    plt.tight_layout()
    plt.show()

In [ ]:
idx='scaled_gaussian_pc20_nc0.60_r0.20'
#results_df.loc[idx]['adata']

In [ ]:
#results_df_lambda05 = results_df.loc[idx]['adata']
label_idx = f'labels_{idx}'
plot_sd_vs_cell_type_composition(adata_banksy, label_idx)

<font size="3"><strong style="color: #c1121f;">**Task 8:** What do you expect would happen when we increase k_geom?</span><br>

<font size="3"><strong style="color: #c1121f;">After answering, Re run Banksy changing k_geom to 80.</span>

<font size="3"><strong style="color: #c1121f;">**Task 9:** What do you expect would happen when we set lambda = 0.99?</span><br>

<font size="3"><strong style="color: #c1121f;">How do you expect the spatial clusters plot to look like?</span>

<font size="3"><strong style="color: #c1121f;">**Task 10:**  How do the Cell Type compositions of each Spatial Domains change as lambda is increasing? Set lambda=0</span><br>

## Describing spatial communities with spatial statistics

<font size="3">Spatial statistics gelfand2010handbook (https://www.taylorfrancis.com/books/mono/10.1201/9781420072884/handbook-spatial-statistics-peter-guttorp-montserrat-fuentes-alan-gelfand-peter-diggle) are quantitative scores that can be used to identify spatial patterns in the tissue. Here, we'll take a look at one spatial statistics implemented in Squidpy with a different set of spatial omics data.

In [ ]:
sd_col = 'labels_scaled_gaussian_pc20_nc0.60_r0.20'
sq.gr.co_occurrence(adata_banksy, cluster_key=sd_col)
sq.pl.co_occurrence(adata_banksy, cluster_key=sd_col, clusters="0", figsize=(8, 5)) 

In [ ]:
sq.gr.co_occurrence(adata_banksy, cluster_key='celltype') 
sq.pl.co_occurrence(adata_banksy, cluster_key='celltype', clusters="Astrocytes", figsize=(8, 5))

Read more: https://squidpy.readthedocs.io/en/stable/

## References

[1] Varrone, M., Tavernari, D., Santamaria-Martínez, A., Walsh, L. A. & Ciriello, G. CellCharter reveals spatial cell niches associated with tissue remodeling and cell plasticity. Nat Genet 56, 74–84 (2024).

[2] Singhal, V. et al. BANKSY unifies cell typing and tissue domain segmentation for scalable spatial omics data analysis. Nat Genet 56, 431–441 (2024).

[3] https://github.com/BrainOmicsCourse/BrainOmics2024/tree/main/3_Day3

[4] https://github.com/NBISweden/workshop-spatial/blob/main/labs/07b_spatial_domains.ipynb

[5] https://github.com/elixir-europe-training/ELIXIR-SCO-spatial-omics/blob/day3-practical-4-cellcharter-init/day_3/practical_4/niche_and_domain_reconstruction.ipynb